# Đọc và tiền xử lý ảnh RGB

Notebook minh họa TASK-01 của My và Huyền trên ảnh thật. Chọn ảnh ở ô cấu hình rồi chạy **Run All**. Đặt `SAVE_PROCESSED = True` để lưu lại bốn tệp của ảnh đã chọn vào `data/processed/`.

In [ ]:
import json
from pathlib import Path
import sys

root = Path.cwd().resolve()
if not (root / "src").is_dir():
    root = root.parent
if not (root / "src").is_dir():
    raise RuntimeError("Mở notebook từ thư mục gốc của repo hoặc notebooks/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from src.config import PROCESSED_DATA_DIR, RAW_DATA_DIR, TARGET_SIZE
from src.data.loader import load_image
from src.data.preprocessing import convert_color_space, flatten_pixels, resize_image

IMAGE_PATH = RAW_DATA_DIR / "image-liquid.jpg"
SAVE_PROCESSED = False

### Đọc ảnh và đưa về RGB

Metadata ghi thứ tự kênh của bộ đọc để chương trình chuyển BGR sang RGB đúng một lần. Ảnh RGB trước khi thu nhỏ được giữ để so sánh ở bước dựng ảnh.

In [ ]:
source_image, source_metadata = load_image(IMAGE_PATH)
original_rgb = convert_color_space(source_image, source_metadata["color_space"], "RGB")
print(f"Ảnh nguồn: {IMAGE_PATH.name} | {source_metadata['color_space']}")
print(f"Ảnh RGB gốc: {original_rgb.shape} | {original_rgb.dtype}")

### Thu nhỏ ảnh và tạo ma trận pixel

Giữ tỷ lệ khung hình và giới hạn cạnh dài nhất ở 256 pixel. Ma trận chứa một hàng `[R, G, B]` cho mỗi pixel, theo thứ tự từ trái sang phải rồi từ trên xuống dưới. Giá trị giữ trong miền 0–255.

In [ ]:
resized_rgb, resized_shape = resize_image(original_rgb, TARGET_SIZE, keep_aspect_ratio=True)
pixel_matrix = flatten_pixels(resized_rgb, as_float=True)

assert pixel_matrix.shape == (resized_shape[0] * resized_shape[1], 3)
assert pixel_matrix.dtype == np.float32
assert np.isfinite(pixel_matrix).all() and 0 <= pixel_matrix.min() <= pixel_matrix.max() <= 255
np.testing.assert_array_equal(pixel_matrix.reshape(resized_rgb.shape), resized_rgb)

fig, axes = plt.subplots(1, 2, figsize=(9, 4), layout="constrained")
for ax, image, label in zip(axes, (original_rgb, resized_rgb), ("RGB gốc", "RGB sau resize")):
    height, width = image.shape[:2]
    ax.imshow(image)
    ax.set_title(f"{label} ({width} × {height})")
    ax.axis("off")
plt.show()
plt.close(fig)

print(f"Ảnh sau resize: {resized_rgb.shape} | {resized_rgb.dtype}")
print(f"Ma trận pixel: {pixel_matrix.shape} | {pixel_matrix.dtype}")

### Metadata và tệp bàn giao

TASK-02 đọc `pixel_matrix.npy` để phân cụm. TASK-03 dùng `original_rgb.png` để so sánh và `resized_shape` trong metadata để dựng lại ảnh từ nhãn cụm. Bật tùy chọn lưu ở ô đầu nếu muốn tạo lại tệp cho ảnh đang chọn.

In [ ]:
metadata = {
    **source_metadata,
    "image_path": IMAGE_PATH.relative_to(root).as_posix(),
    "source_color_space": source_metadata["color_space"],
    "color_space": "RGB",
    "resized_shape": resized_shape,
    "image_dtype": str(resized_rgb.dtype),
    "dtype": str(pixel_matrix.dtype),
    "value_range": [0, 255],
    "observed_range": [float(pixel_matrix.min()), float(pixel_matrix.max())],
}
print(json.dumps(metadata, ensure_ascii=False, indent=2))

if SAVE_PROCESSED:
    output = PROCESSED_DATA_DIR / IMAGE_PATH.stem
    output.mkdir(parents=True, exist_ok=True)
    Image.fromarray(original_rgb).save(output / "original_rgb.png")
    Image.fromarray(resized_rgb).save(output / "resized_rgb.png")
    np.save(output / "pixel_matrix.npy", pixel_matrix, allow_pickle=False)
    (output / "metadata.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )